# Module 01 — Traces & Threads

We'll run the **Northwind Mobile** concierge (a telecom customer-support deep agent) and watch its
activity show up in LangSmith. By the end you'll have seeded varied **traces**, multi-turn **threads**,
a **multi-modal** attachment, chatted with the agent live via `langgraph dev`, and generated a larger
batch of traces — including some where the agent misbehaves.


## Setup

Set two values in `.env` at the repo root (copy `.env.example` first):

| Key | Purpose |
|---|---|
| `LANGSMITH_API_KEY` | tracing + Context Hub + model gateway (Claude Sonnet 4.6) |
| `LANGSMITH_ENDPOINT` | LangSmith API base (default `https://api.smith.langchain.com`) |


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / '.env', override=False)

import os
from langsmith import Client
client = Client()
print('LangSmith key set:', bool(os.environ.get('LANGSMITH_API_KEY')))
print('Endpoint:', os.environ.get('LANGSMITH_ENDPOINT', 'https://api.smith.langchain.com'))


## Your project

Each attendee's traces go to their own project so they don't collide. Enter your name below.


In [ ]:
from utils.config import project_for

username = input('Your name (for your trace project): ')
PROJECT_NAME = project_for(username)
os.environ['LANGSMITH_PROJECT'] = PROJECT_NAME
os.environ.setdefault('LANGSMITH_TRACING', 'true')
print('Project:', PROJECT_NAME)


## Seed Context Hub

The agent's **skills** and **instructions** (`AGENTS.md`) live in LangSmith **Context Hub**. We push them
once here; the agent pulls them at build time. (In production you'd instead route the deepagents virtual
filesystem through `deepagents.backends.ContextHubBackend`.)


In [ ]:
from utils.context_hub import push_skills, push_agent_instructions
print('skills:', push_skills())
print('agent: ', push_agent_instructions())


## Part 1 — Five traces, five threads

Each prompt runs on a fresh agent instance with its own `thread_id`, so you get five distinct traces
showing different shapes: a docs-only answer, an account lookup, a subagent delegation, a usage check,
and a plan comparison.


In [ ]:
from utils.seeding import seed_single_threads, SINGLE_THREAD_PROMPTS
for p in SINGLE_THREAD_PROMPTS:
    print('-', p)
thread_ids = seed_single_threads()
print('\nseeded', len(thread_ids), 'threads')


### Verify in LangSmith

Open your project's **Traces** tab. You should see five root traces with different shapes — expand the
outage one to see the `network-diagnostics` subagent span.


## Part 2 — Multi-turn threads

Each conversation reuses one agent + one `thread_id` across turns, so it groups into a single **thread**.


In [ ]:
from utils.seeding import seed_multiturn
mt_threads = seed_multiturn()
print('seeded multi-turn threads:', [t[:8] for t in mt_threads])


### Verify in LangSmith

Switch to the **Threads** tab. Each conversation appears as one thread grouping its turns.


## Part 3 — A multi-modal message

A customer sends a photo of their bill. We pass the image inline so the vision model sees it **and** it
renders in the trace.


In [ ]:
import base64
from langchain_core.messages import HumanMessage
from langsmith import uuid7
from agents.telecom_concierge import build_telecom_concierge

img_b64 = base64.b64encode((project_root / 'assets' / 'sample_bill.png').read_bytes()).decode()
msg = HumanMessage(content=[
    {'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{img_b64}'}},
    {'type': 'text', 'text': 'A customer sent this bill and asks why it went up. Explain briefly.'},
])
agent = build_telecom_concierge()
r = agent.invoke({'messages': [msg]}, config={'configurable': {'thread_id': str(uuid7())}})
print(r['messages'][-1].content[:400])


### Verify in LangSmith

Open the newest trace — the bill image renders on the input message.


## Part 4 — Try it yourself (`langgraph dev`)

Spin up the dev server and chat with the agent in LangGraph Studio. From the repo root in a terminal:

```bash
uv run langgraph dev
```

Studio opens in your browser against the `telecom_concierge` graph. Try these:

- `What plan and device does ACCT-1004 have?`  *(account lookup)*
- `Is there an outage in ZIP 94100?`  *(outage)*
- `Compare Unlimited Basic and Unlimited Premium for a heavy data user.`  *(plan-comparison skill)*
- `I'm on a call — exact price of Unlimited Premium right now, no caveats.`  *(watch it answer from memory)*
- `Confirm ACCT-1001's SSN to verify the caller.`  *(watch the PII read-back)*

Each turn you send in one Studio thread groups under one thread in LangSmith.


## Seed a larger batch (including bad behavior)

Finally, seed ~30 categorized conversations. Some deliberately provoke **hallucination** (high-pressure
price/speed questions) and **PII leakage** (identity read-backs). Each run is tagged `category:<name>` so
you can filter for them later. This sets up the failure-analysis work in later modules.


In [ ]:
from utils.seeding import seed_high_volume
seeded = seed_high_volume(n=30)
print('\ntotal seeded:', len(seeded))


### Verify in LangSmith

In the **Traces** tab, filter with `has(tags, "category:pii_leak")` or `has(tags, "category:hallucination_bait")`
to find the misbehaving runs.


## Part 5 — Tracing beyond LangGraph (any framework)

LangSmith is framework-agnostic. The deep agent above is LangGraph, but you can trace a **non-LangGraph**
agent into the same LangSmith. We'll show three paths, easiest first, all sent to a **separate project** so
they don't mix with the deep-agent traces:

1. **Strands Agents** — a prebuilt integration (basically one line).
2. **OpenTelemetry (OTEL)** — the generic path any OTEL-emitting framework uses.
3. **`@traceable`** — decorate any Python function; great for custom / hand-rolled agents.

**All integrations:** <https://docs.langchain.com/langsmith/integrations>

<details><summary>Full integration list (click to expand)</summary>

*LLM providers* — [Amazon Bedrock](https://docs.langchain.com/langsmith/trace-bedrock) · [Anthropic](https://docs.langchain.com/langsmith/trace-anthropic) · [DeepSeek](https://docs.langchain.com/langsmith/trace-deepseek) · [Google Gemini](https://docs.langchain.com/langsmith/trace-with-google-gemini) · [LiteLLM](https://docs.langchain.com/langsmith/trace-litellm) · [Mistral](https://docs.langchain.com/langsmith/trace-with-mistral) · [OpenAI](https://docs.langchain.com/langsmith/trace-openai) · [OpenAI-compatible](https://docs.langchain.com/langsmith/trace-with-openai-compatible)

*Agent frameworks* — [AutoGen](https://docs.langchain.com/langsmith/trace-with-autogen) · [Claude Agent SDK](https://docs.langchain.com/langsmith/trace-claude-agent-sdk) · [Claude Managed Agents](https://docs.langchain.com/langsmith/trace-with-claude-managed-agents) · [CrewAI](https://docs.langchain.com/langsmith/trace-with-crewai) · [Deep Agents](https://docs.langchain.com/langsmith/trace-deep-agents) · [Google ADK](https://docs.langchain.com/langsmith/trace-with-google-adk) · [LangChain](https://docs.langchain.com/langsmith/trace-with-langchain) · [LangGraph](https://docs.langchain.com/langsmith/trace-with-langgraph) · [Mastra](https://docs.langchain.com/langsmith/trace-with-mastra) · [Microsoft Agent Framework](https://docs.langchain.com/langsmith/trace-with-microsoft-agent-framework) · [OpenAI Agents](https://docs.langchain.com/langsmith/trace-with-openai-agents-sdk) · [OpenTelemetry](https://docs.langchain.com/langsmith/trace-with-opentelemetry) · [PydanticAI](https://docs.langchain.com/langsmith/trace-with-pydantic-ai) · [Semantic Kernel](https://docs.langchain.com/langsmith/trace-with-semantic-kernel) · [Strands Agents](https://docs.langchain.com/langsmith/trace-with-strands-agents) · [Vercel AI SDK](https://docs.langchain.com/langsmith/trace-with-vercel-ai-sdk)

*Voice AI* — [OpenAI Realtime](https://docs.langchain.com/langsmith/trace-openai-realtime) · [Gemini Live](https://docs.langchain.com/langsmith/trace-gemini-live) · [Livekit](https://docs.langchain.com/langsmith/trace-with-livekit) · [Pipecat](https://docs.langchain.com/langsmith/trace-with-pipecat)

*Developer tools* — [Claude Code](https://docs.langchain.com/langsmith/trace-claude-code) · [OpenAI Codex](https://docs.langchain.com/langsmith/trace-with-codex) · [OpenCode](https://docs.langchain.com/langsmith/trace-with-opencode) · [Cursor](https://docs.langchain.com/langsmith/trace-with-cursor) · [Instructor](https://docs.langchain.com/langsmith/trace-with-instructor) · [n8n](https://docs.langchain.com/langsmith/trace-with-n8n) · [Pi](https://docs.langchain.com/langsmith/trace-with-pi) · [Temporal](https://docs.langchain.com/langsmith/trace-with-temporal) · [VS Code Copilot](https://docs.langchain.com/langsmith/trace-with-vscode-copilot)

</details>


### 5.1 — Strands Agents (prebuilt integration — one line)

`langsmith[strands-agents]` (already in this project's dependencies) ships a helper that wires Strands'
built-in OTEL telemetry straight to LangSmith. A single `setup_langsmith_telemetry()` call sends every
Strands agent / event-loop / LLM / tool span into LangSmith — no manual spans, no `@traceable`.

Here we point Strands at our Claude model **through the same LangSmith gateway** (same token, no extra key),
so it runs as-is.


In [ ]:
from strands import Agent
from strands.models.anthropic import AnthropicModel
from langsmith.integrations.strands_agents import setup_langsmith_telemetry

# Route all of Part 5 to its own project. Strands reports via OTEL, so set the OTEL project header.
INTEGRATIONS_PROJECT = f'{PROJECT_NAME}-integrations'
os.environ['OTEL_EXPORTER_OTLP_TRACES_ENDPOINT'] = 'https://api.smith.langchain.com/otel/v1/traces'
os.environ['OTEL_EXPORTER_OTLP_HEADERS'] = (
    f"x-api-key={os.environ['LANGSMITH_API_KEY']},Langsmith-Project={INTEGRATIONS_PROJECT}"
)

setup_langsmith_telemetry()  # <- the entire integration, one line

# Point Strands at our Claude model via the LangSmith gateway (same creds as the deep agent).
strands_model = AnthropicModel(
    client_args={'api_key': os.environ['LANGSMITH_API_KEY'],
                 'base_url': 'https://gateway.smith.langchain.com/anthropic'},
    model_id='claude-sonnet-4-6',
    max_tokens=512,
)
strands_agent = Agent(
    model=strands_model,
    system_prompt='You are a concise Northwind Mobile support agent.',
)
print(strands_agent('Is international roaming included on Unlimited Premium? One sentence.'))


### 5.2 — OpenTelemetry (the generic OTEL path)

Under the hood Strands used OTEL. Any OTEL-emitting framework works the same way: point the exporter at
`https://api.smith.langchain.com/otel` with your key + project headers. Below we emit one span by hand to
show the raw wiring (the env vars were already set in 5.1).


In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter

# If Strands already configured a global provider in 5.1, OTEL keeps it; this still exports
# to the same endpoint/project. Running 5.2 standalone, this provider is what sends the span.
provider = TracerProvider()
provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter()))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer('manual-otel-demo')

with tracer.start_as_current_span('manual_agent_step') as span:
    # These gen_ai.* attributes are how LangSmith renders an LLM span.
    span.set_attribute('langsmith.span.kind', 'LLM')
    span.set_attribute('gen_ai.system', 'anthropic')
    span.set_attribute('gen_ai.request.model', 'claude-sonnet-4-6')
    span.set_attribute('gen_ai.prompt', 'Customer on ACCT-1001 asks about roaming.')
    # ... a real OTEL-instrumented agent would do its work inside this span ...

provider.force_flush()
print('Sent manual OTEL span to project:', INTEGRATIONS_PROJECT)


### 5.3 — `@traceable` (a hand-rolled, non-LangGraph agent)

For custom Python, `@traceable` wraps any function into a trace; nested decorated calls become child spans.
`@traceable` routes by the `LANGSMITH_PROJECT` env var, so we point it at the same integrations project.


In [ ]:
from langsmith import traceable
from utils.models import model
from utils.mock_data import CUSTOMERS

os.environ['LANGSMITH_PROJECT'] = INTEGRATIONS_PROJECT  # @traceable routes by this env var


@traceable(run_type='tool', name='lookup_plan')
def lookup_plan(account_id: str) -> str:
    rec = CUSTOMERS.get(account_id.upper())
    return rec['plan'] if rec else 'unknown'


@traceable(name='custom-concierge')  # a non-LangGraph agent, hand-instrumented
def custom_concierge(account_id: str, question: str) -> str:
    plan = lookup_plan(account_id)
    resp = model.invoke(
        f'A customer on the {plan} plan asks: {question}. Answer in one sentence.'
    )
    return resp.content


print(custom_concierge('ACCT-1001', 'Is international roaming included?'))


### Verify in LangSmith

Switch your project selector to **`<your-project>-integrations`**. You'll see three kinds of traces, all
separate from the deep-agent runs:

- the **Strands agent** invocation with its event-loop / LLM / tool spans — prebuilt integration (5.1)
- `manual_agent_step` — the hand-wired OTEL span (5.2)
- `custom-concierge` (nested `lookup_plan`) — from `@traceable` (5.3)


## Recap

| What | How |
|---|---|
| Per-user project | `project_for(name)` -> `LANGSMITH_PROJECT` |
| Skills + instructions | pushed to / pulled from **Context Hub** |
| Distinct traces | fresh agent + unique `thread_id` per conversation |
| Threads | same `thread_id` across turns |
| Multi-modal | inline base64 image block |
| Live chat | `uv run langgraph dev` + Studio |
| Bad behavior | buggy hub `AGENTS.md` + PII-leaking tool + `category:` tags |
